# 10 - Multi-Seed Training

Bu notebook, tezdeki **tek random seed sınırlılığını** azaltmak için fine-tune edilen modelleri
iki yeni seed ile tekrar eğitir:

- `123`
- `2024`

Mevcut `seed=42` deneyleri tekrar edilmez.

## Modeller

- BERT-base-uncased
- DistilBERT-base-uncased
- RoBERTa-base
- Fine-tuned ProsusAI/FinBERT

## Deney düzeni

`04_train_plain_sentiment_models.ipynb` ve `08_finetune_finbert_target_dataset.ipynb`
ile aynı temel eğitim ayarları kullanılır:

- epochs = 4
- learning rate = 2e-5
- train batch = 16
- eval batch = 32
- max length = 128
- weight decay = 0.01
- class weights = açık
- model seçimi = validation Macro-F1

Toplam **8 yeni eğitim koşusu** vardır: `4 model × 2 seed`.

Her koşu kendi checkpoint klasörüne yazılır. Notebook yarıda kesilirse tekrar çalıştırıldığında:
- tamamlanmış run'lar atlanır,
- yarım run'lar son checkpoint'ten devam eder.

Ayrıca her run için test prediction CSV'si kaydedilir. Bunlar daha sonra
istatistiksel test ve hata analizi notebooklarında kullanılacaktır.


In [ ]:
from thesis_utils import PREVIEW_ROWS, PROJECT_ROOT, paths

from pathlib import Path
import os
import sys
import gc
import json
import random
import inspect
import subprocess
import warnings

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ============================================================
# 1) AYARLAR
# ============================================================
SEEDS_TO_RUN = [123, 2024]

MAX_LENGTH = 128
EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 2
USE_CLASS_WEIGHTS = True

# True yaparsan tamamlanmış run'ları da baştan eğitir.
FORCE_RETRAIN = False

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}
ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}
LABELS_ORDER = ["negative", "neutral", "positive"]
NUM_LABELS = 3

TEXT_COL = "input_text"
LABEL_COL = "label"
LABEL_ID_COL = "label_id"

MODEL_CONFIGS = [
    {
        "model_key": "bert",
        "display_name": "BERT-base-uncased",
        "model_name": "bert-base-uncased",
        "is_finbert": False,
    },
    {
        "model_key": "distilbert",
        "display_name": "DistilBERT-base-uncased",
        "model_name": "distilbert-base-uncased",
        "is_finbert": False,
    },
    {
        "model_key": "roberta",
        "display_name": "RoBERTa-base",
        "model_name": "roberta-base",
        "is_finbert": False,
    },
    {
        "model_key": "finbert",
        "display_name": "Fine-tuned FinBERT",
        "model_name": "ProsusAI/finbert",
        "is_finbert": True,
    },
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch :", torch.__version__)
print("Seeds :", SEEDS_TO_RUN)
print("Runs  :", len(MODEL_CONFIGS) * len(SEEDS_TO_RUN))


# ============================================================
# 2) PATH'LER
# ============================================================
SPLIT_DIR = paths.PLAIN_SENTIMENT_SPLIT_V1_DIR
TRAIN_PATH = SPLIT_DIR / "train_df.parquet"
VAL_PATH = SPLIT_DIR / "val_df.parquet"
TEST_PATH = SPLIT_DIR / "test_df.parquet"

BASE_CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints" / "financial_sentiment_multi_model"
MULTISEED_ROOT = BASE_CHECKPOINT_ROOT / "multiseed"
MULTISEED_ROOT.mkdir(parents=True, exist_ok=True)

AGGREGATE_RESULTS_PATH = MULTISEED_ROOT / "multiseed_results_new_seeds.csv"

print("\nSplit dir     :", SPLIT_DIR)
print("Multiseed root:", MULTISEED_ROOT)


In [ ]:
# ============================================================
# 3) MEVCUT SPLITLER
# ============================================================
for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Split bulunamadı: {path}\n"
            "Önce 04_train_plain_sentiment_models.ipynb içindeki split üretimini çalıştır."
        )

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for col in [TEXT_COL, LABEL_COL, LABEL_ID_COL]:
        if col not in df.columns:
            raise ValueError(f"{split_name} splitinde eksik kolon: {col}")

    df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()
    df[LABEL_ID_COL] = df[LABEL_ID_COL].astype(int)

    inconsistent = df[
        df.apply(lambda row: LABEL2ID[row[LABEL_COL]] != int(row[LABEL_ID_COL]), axis=1)
    ]
    if len(inconsistent):
        raise ValueError(
            f"{split_name}: label-label_id uyumsuzluğu bulundu: {len(inconsistent)}"
        )

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts().sort_index())

# 04 notebookundaki dağılımla eşleşmesi beklenir:
# train=12950, val=1432, test=2386


In [ ]:
# ============================================================
# 4) YARDIMCI FONKSİYONLAR
# ============================================================

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism mümkün olduğunca güçlendirilir.
    # CPU'da çoğu işlem zaten deterministik olsa da kayıt altına alınır.
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def prepare_hf_dataset(split_df, tokenizer, max_length=128):
    temp = split_df[[TEXT_COL, LABEL_ID_COL]].copy()
    temp[TEXT_COL] = temp[TEXT_COL].astype(str)
    temp[LABEL_ID_COL] = temp[LABEL_ID_COL].astype(int)
    temp = temp.rename(columns={LABEL_ID_COL: "labels"})

    ds = Dataset.from_pandas(temp.reset_index(drop=True))

    def tokenize_fn(batch):
        return tokenizer(
            batch[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    ds = ds.map(tokenize_fn, batched=True)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in ds.column_names:
        keep_cols.append("token_type_ids")

    ds.set_format(type="torch", columns=keep_cols)
    return ds


def trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)

    if "processing_class" in sig.parameters:
        return {"processing_class": tokenizer}
    if "tokenizer" in sig.parameters:
        return {"tokenizer": tokenizer}

    return {}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


def build_training_args(output_dir, seed):
    sig = inspect.signature(TrainingArguments.__init__)

    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        report_to="none",
        seed=seed,
    )

    if "data_seed" in sig.parameters:
        kwargs["data_seed"] = seed

    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "epoch"

    if "save_strategy" in sig.parameters:
        kwargs["save_strategy"] = "epoch"

    if "save_total_limit" in sig.parameters:
        kwargs["save_total_limit"] = SAVE_TOTAL_LIMIT

    if "load_best_model_at_end" in sig.parameters:
        kwargs["load_best_model_at_end"] = True

    if "metric_for_best_model" in sig.parameters:
        kwargs["metric_for_best_model"] = "f1_macro"

    if "greater_is_better" in sig.parameters:
        kwargs["greater_is_better"] = True

    if "logging_strategy" in sig.parameters:
        kwargs["logging_strategy"] = "steps"

    if "fp16" in sig.parameters:
        kwargs["fp16"] = torch.cuda.is_available()

    return TrainingArguments(**kwargs)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights_tensor is not None:
            weights = self.class_weights_tensor.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )

        return (loss, outputs) if return_outputs else loss


def calculate_class_weights(df):
    y = df[LABEL_ID_COL].astype(int).to_numpy()
    counts = np.bincount(y, minlength=NUM_LABELS)

    weights = np.array([
        len(y) / (NUM_LABELS * count) if count > 0 else 1.0
        for count in counts
    ], dtype=np.float32)

    return torch.tensor(weights, dtype=torch.float)


CLASS_WEIGHTS = calculate_class_weights(train_df) if USE_CLASS_WEIGHTS else None

print("Class weights:", CLASS_WEIGHTS)


In [ ]:
# ============================================================
# 5) FİN BERT LABEL HEAD HİZALAMA
# ============================================================

def normalize_finbert_label(value):
    value = str(value).lower().strip()

    mapping = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "label_0": "positive",
        "label_1": "negative",
        "label_2": "neutral",
    }

    return mapping.get(value, value)


def align_finbert_classifier_to_project_labels(model):
    """
    ProsusAI/finbert'in mevcut sentiment classifier ağırlıklarını korur,
    fakat çıktı satırlarını projenin sabit sınıf sırasına taşır:

      0 = negative
      1 = neutral
      2 = positive

    Sadece config.label2id değiştirmek yeterli değildir; classifier ağırlık
    satırlarının da aynı sıraya getirilmesi gerekir.
    """
    original_id2label = {
        int(k): normalize_finbert_label(v)
        for k, v in model.config.id2label.items()
    }

    if set(original_id2label.values()) != set(LABEL2ID.keys()):
        raise ValueError(
            "FinBERT label mapping beklenenden farklı: "
            f"{original_id2label}"
        )

    if not hasattr(model, "classifier"):
        raise AttributeError("FinBERT modelinde classifier katmanı bulunamadı.")

    classifier = model.classifier
    old_weight = classifier.weight.detach().clone()
    old_bias = (
        classifier.bias.detach().clone()
        if classifier.bias is not None
        else None
    )

    source_id_for_label = {
        label: source_id
        for source_id, label in original_id2label.items()
    }

    with torch.no_grad():
        for target_id, target_label in ID2LABEL.items():
            source_id = source_id_for_label[target_label]
            classifier.weight[target_id].copy_(old_weight[source_id])

            if old_bias is not None:
                classifier.bias[target_id].copy_(old_bias[source_id])

    model.config.id2label = ID2LABEL.copy()
    model.config.label2id = LABEL2ID.copy()

    return model


def load_initial_model(model_cfg):
    model_name = model_cfg["model_name"]

    if model_cfg["is_finbert"]:
        # FinBERT'in hazır finansal sentiment başlığı korunur.
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        model = align_finbert_classifier_to_project_labels(model)
    else:
        # 04 notebookundaki genel amaçlı model yaklaşımı.
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        )

    model.config.id2label = ID2LABEL.copy()
    model.config.label2id = LABEL2ID.copy()

    return model


In [ ]:
# ============================================================
# 6) DEĞERLENDİRME VE KAYIT
# ============================================================

def softmax_numpy(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def evaluate_and_save(
    trainer,
    dataset,
    source_df,
    split_name,
    model_cfg,
    seed,
    results_dir,
):
    pred_output = trainer.predict(dataset)

    logits = pred_output.predictions
    y_true = pred_output.label_ids.astype(int)

    probs = softmax_numpy(logits)
    y_pred = np.argmax(probs, axis=1).astype(int)
    confidence = probs.max(axis=1)

    # ============================================================
    # GÜVENLİK KONTROLÜ
    # ============================================================

    if len(source_df) != len(y_true):
        raise ValueError(
            f"{split_name}: source_df ve prediction uzunlukları eşleşmiyor. "
            f"source_df={len(source_df)}, predictions={len(y_true)}"
        )

    # ============================================================
    # METRİKLER
    # ============================================================

    precision_macro, recall_macro, _, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        )
    )

    per_class_precision, per_class_recall, per_class_f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average=None,
            zero_division=0,
        )
    )

    metrics = {
        "model_key": model_cfg["model_key"],
        "model": model_cfg["display_name"],
        "model_name": model_cfg["model_name"],
        "seed": int(seed),
        "split": split_name,
        "n_eval": int(len(y_true)),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "precision_macro": float(
            precision_macro
        ),

        "recall_macro": float(
            recall_macro
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),

        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="weighted",
                zero_division=0,
            )
        ),

        "precision_negative": float(
            per_class_precision[0]
        ),
        "recall_negative": float(
            per_class_recall[0]
        ),
        "f1_negative": float(
            per_class_f1[0]
        ),
        "support_negative": int(
            support[0]
        ),

        "precision_neutral": float(
            per_class_precision[1]
        ),
        "recall_neutral": float(
            per_class_recall[1]
        ),
        "f1_neutral": float(
            per_class_f1[1]
        ),
        "support_neutral": int(
            support[1]
        ),

        "precision_positive": float(
            per_class_precision[2]
        ),
        "recall_positive": float(
            per_class_recall[2]
        ),
        "f1_positive": float(
            per_class_f1[2]
        ),
        "support_positive": int(
            support[2]
        ),
    }

    # ============================================================
    # CONFUSION MATRIX
    # ============================================================

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2],
    )

    row_totals = cm.sum(
        axis=1,
        keepdims=True,
    )

    cm_norm = np.divide(
        cm,
        row_totals,
        out=np.zeros_like(
            cm,
            dtype=float,
        ),
        where=row_totals != 0,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"true_{x}"
            for x in LABELS_ORDER
        ],
        columns=[
            f"pred_{x}"
            for x in LABELS_ORDER
        ],
    )

    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=[
            f"true_{x}"
            for x in LABELS_ORDER
        ],
        columns=[
            f"pred_{x}"
            for x in LABELS_ORDER
        ],
    )

    # ============================================================
    # PREDICTION DATAFRAME
    # ============================================================

    pred_df = (
        source_df
        .reset_index(drop=True)
        .copy()
    )

    # ------------------------------------------------------------
    # sample_id
    # ------------------------------------------------------------
    # Split zaten sample_id taşıyorsa onu koruyoruz.
    # Yoksa deterministic ID üretiyoruz.
    # Bu ID daha sonra McNemar / bootstrap eşleştirmesinde kullanılacak.
    # ------------------------------------------------------------

    if "sample_id" not in pred_df.columns:
        pred_df.insert(
            0,
            "sample_id",
            [
                f"{split_name}_{i:06d}"
                for i in range(
                    len(pred_df)
                )
            ],
        )

        print(
            f"{split_name}: sample_id yoktu, "
            "otomatik oluşturuldu."
        )

    else:
        pred_df["sample_id"] = (
            pred_df["sample_id"]
            .astype(str)
            .str.strip()
        )

        print(
            f"{split_name}: mevcut sample_id "
            "değerleri korunuyor."
        )

    # ------------------------------------------------------------
    # Boş ID kontrolü
    # ------------------------------------------------------------

    invalid_sample_id = (
        pred_df["sample_id"].isna()
        |
        pred_df["sample_id"]
        .astype(str)
        .str.strip()
        .isin([
            "",
            "nan",
            "None",
            "none",
        ])
    )

    if invalid_sample_id.any():
        raise ValueError(
            f"{split_name}: "
            f"{int(invalid_sample_id.sum())} "
            "geçersiz sample_id bulundu."
        )

    # ------------------------------------------------------------
    # Duplicate ID kontrolü
    # ------------------------------------------------------------

    duplicate_count = int(
        pred_df["sample_id"]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{split_name}: sample_id içinde "
            f"{duplicate_count} duplicate değer bulundu."
        )

    # ============================================================
    # GOLD / PREDICTION / PROBABILITY
    # ============================================================

    pred_df["gold_label_id"] = (
        y_true
    )

    pred_df["gold_label"] = [
        ID2LABEL[int(x)]
        for x in y_true
    ]

    pred_df["prediction_id"] = (
        y_pred
    )

    pred_df["prediction"] = [
        ID2LABEL[int(x)]
        for x in y_pred
    ]

    pred_df["correct"] = (
        y_true == y_pred
    )

    pred_df["confidence"] = (
        confidence
    )

    pred_df["prob_negative"] = (
        probs[:, 0]
    )

    pred_df["prob_neutral"] = (
        probs[:, 1]
    )

    pred_df["prob_positive"] = (
        probs[:, 2]
    )

    # ============================================================
    # LOGITLER
    # ============================================================
    # Daha sonra calibration / ECE için tekrar inference
    # almak zorunda kalmamak için saklıyoruz.

    pred_df["logit_negative"] = (
        logits[:, 0]
    )

    pred_df["logit_neutral"] = (
        logits[:, 1]
    )

    pred_df["logit_positive"] = (
        logits[:, 2]
    )

    # ============================================================
    # HATA YÖNÜ
    # ============================================================

    pred_df["error_transition"] = (
        np.where(
            pred_df["correct"],
            "correct",
            (
                pred_df["gold_label"]
                + " -> "
                + pred_df["prediction"]
            ),
        )
    )

    # ============================================================
    # CLASSIFICATION REPORT
    # ============================================================

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=LABELS_ORDER,
        output_dict=True,
        zero_division=0,
    )

    # ============================================================
    # KLASÖR
    # ============================================================

    results_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ============================================================
    # DOSYA YOLLARI
    # ============================================================

    metrics_path = (
        results_dir
        / f"{split_name}_metrics.json"
    )

    pred_path = (
        results_dir
        / f"{split_name}_predictions.csv"
    )

    cm_path = (
        results_dir
        / f"{split_name}_confusion_matrix.csv"
    )

    cm_norm_path = (
        results_dir
        / f"{split_name}_confusion_matrix_normalized.csv"
    )

    report_path = (
        results_dir
        / f"{split_name}_classification_report.csv"
    )

    # ============================================================
    # KAYDET
    # ============================================================

    with open(
        metrics_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            metrics,
            f,
            indent=2,
            ensure_ascii=False,
        )

    pred_df.to_csv(
        pred_path,
        index=False,
        encoding="utf-8-sig",
    )

    cm_df.to_csv(
        cm_path,
        encoding="utf-8-sig",
    )

    cm_norm_df.to_csv(
        cm_norm_path,
        encoding="utf-8-sig",
    )

    pd.DataFrame(
        report
    ).T.to_csv(
        report_path,
        encoding="utf-8-sig",
    )

    # ============================================================
    # EKRANA YAZ
    # ============================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{model_cfg['display_name']} "
        f"| seed={seed} "
        f"| {split_name.upper()}"
    )

    print(
        "=" * 100
    )

    print(
        f"N               : "
        f"{len(y_true)}"
    )

    print(
        f"Accuracy        : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Macro Precision : "
        f"{metrics['precision_macro']:.4f}"
    )

    print(
        f"Macro Recall    : "
        f"{metrics['recall_macro']:.4f}"
    )

    print(
        f"Macro F1        : "
        f"{metrics['f1_macro']:.4f}"
    )

    print(
        f"Weighted F1     : "
        f"{metrics['f1_weighted']:.4f}"
    )

    print("\nClass F1:")

    print(
        f"Negative        : "
        f"{metrics['f1_negative']:.4f}"
    )

    print(
        f"Neutral         : "
        f"{metrics['f1_neutral']:.4f}"
    )

    print(
        f"Positive        : "
        f"{metrics['f1_positive']:.4f}"
    )

    print(
        "\nPrediction file:",
        pred_path,
    )

    return metrics

In [ ]:
# ============================================================
# 7) TEK MODEL / TEK SEED DENEY?
# ============================================================

def make_run_paths(model_cfg, seed):
    run_dir = MULTISEED_ROOT / f"{model_cfg['model_key']}_seed{seed}"
    checkpoint_dir = run_dir / "checkpoints"
    final_model_dir = run_dir / "final_model"
    results_dir = run_dir / "results"

    return run_dir, checkpoint_dir, final_model_dir, results_dir


def load_metrics_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def run_single_experiment(model_cfg, seed):
    set_all_seeds(seed)

    run_dir, checkpoint_dir, final_model_dir, results_dir = make_run_paths(
        model_cfg,
        seed,
    )

    test_metrics_path = results_dir / "test_metrics.json"

    if test_metrics_path.exists() and not FORCE_RETRAIN:
        print(
            f"ATLANIYOR: {model_cfg['display_name']} | seed={seed} "
            "test_metrics.json zaten var."
        )
        return load_metrics_json(test_metrics_path)

    tokenizer_source = (
        final_model_dir
        if final_model_dir.exists() and not FORCE_RETRAIN
        else model_cfg["model_name"]
    )

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)

    train_ds = prepare_hf_dataset(train_df, tokenizer, MAX_LENGTH)
    val_ds = prepare_hf_dataset(val_df, tokenizer, MAX_LENGTH)
    test_ds = prepare_hf_dataset(test_df, tokenizer, MAX_LENGTH)

    if final_model_dir.exists() and not FORCE_RETRAIN:
        print(
            f"MEVCUT MODEL KULLANILIYOR: {model_cfg['display_name']} | seed={seed}"
        )
        model = AutoModelForSequenceClassification.from_pretrained(final_model_dir)
        model.config.id2label = ID2LABEL.copy()
        model.config.label2id = LABEL2ID.copy()
    else:
        print(
            "\n" + "=" * 110 + "\n"
            f"E??T?M: {model_cfg['display_name']} | seed={seed}\n"
            + "=" * 110
        )

        model = load_initial_model(model_cfg)

    training_args = build_training_args(checkpoint_dir, seed)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        class_weights_tensor=CLASS_WEIGHTS,
        **trainer_tokenizer_kwargs(tokenizer),
    )

    if not final_model_dir.exists() or FORCE_RETRAIN:
        last_checkpoint = None
        if checkpoint_dir.exists() and not FORCE_RETRAIN:
            last_checkpoint = get_last_checkpoint(str(checkpoint_dir))

        trainer.train(resume_from_checkpoint=last_checkpoint)

        final_model_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(final_model_dir))
        tokenizer.save_pretrained(str(final_model_dir))

    val_metrics = evaluate_and_save(
        trainer,
        val_ds,
        val_df,
        "val",
        model_cfg,
        seed,
        results_dir,
    )

    test_metrics = evaluate_and_save(
        trainer,
        test_ds,
        test_df,
        "test",
        model_cfg,
        seed,
        results_dir,
    )

    run_summary = {
        "model_key": model_cfg["model_key"],
        "model": model_cfg["display_name"],
        "model_name": model_cfg["model_name"],
        "seed": int(seed),
        "run_dir": str(run_dir),
        "val_accuracy": val_metrics["accuracy"],
        "val_f1_macro": val_metrics["f1_macro"],
        "test_accuracy": test_metrics["accuracy"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
    }

    with open(results_dir / "run_summary.json", "w", encoding="utf-8") as f:
        json.dump(run_summary, f, indent=2, ensure_ascii=False)

    del trainer, model, tokenizer, train_ds, val_ds, test_ds
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return test_metrics


In [ ]:
# ============================================================
# 8) TÜM YENİ SEED RUN'LARINI ÇALIŞTIR
# ============================================================

all_test_metrics = []

for seed in SEEDS_TO_RUN:
    for model_cfg in MODEL_CONFIGS:
        try:
            metrics = run_single_experiment(model_cfg, seed)
            all_test_metrics.append(metrics)

            # Her run sonunda ara sonuç kaydet.
            pd.DataFrame(all_test_metrics).to_csv(
                AGGREGATE_RESULTS_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        except Exception as exc:
            print("\n" + "!" * 110)
            print(
                f"HATA: {model_cfg['display_name']} | seed={seed}\n"
                f"{type(exc).__name__}: {exc}"
            )
            print("Notebook durduruluyor; hata giderildikten sonra tekrar Run All yapabilirsin.")
            print("Tamamlanmış run'lar tekrar eğitilmeyecek.")
            print("!" * 110)
            raise

new_seed_results_df = pd.DataFrame(all_test_metrics)

print("\n" + "=" * 110)
print("YENİ SEED TEST SONUÇLARI")
print("=" * 110)

display(
    new_seed_results_df[
        [
            "model",
            "seed",
            "accuracy",
            "f1_macro",
            "f1_weighted",
            "f1_negative",
            "f1_neutral",
            "f1_positive",
        ]
    ].round(6)
)


In [ ]:
# ============================================================
# 9) SEED=42 REFERANS SONUÇLARINI EKLE
# ============================================================
# BERT / DistilBERT / RoBERTa değerleri 04 notebookundaki tamamlanmış
# seed=42 test sonuçlarıdır.
#
# Fine-tuned FinBERT seed=42 değeri 08 notebookunun ürettiği summary
# dosyasından okunur. Böylece FinBERT için sonuç uydurulmaz.

historical_seed42 = [
    {
        "model_key": "bert",
        "model": "BERT-base-uncased",
        "seed": 42,
        "accuracy": 0.8722,
        "f1_macro": 0.8378,
        "f1_weighted": 0.8733,
    },
    {
        "model_key": "distilbert",
        "model": "DistilBERT-base-uncased",
        "seed": 42,
        "accuracy": 0.8583,
        "f1_macro": 0.8218,
        "f1_weighted": 0.8593,
    },
    {
        "model_key": "roberta",
        "model": "RoBERTa-base",
        "seed": 42,
        "accuracy": 0.8906,
        "f1_macro": 0.8651,
        "f1_weighted": 0.8918,
    },
]

seed42_df = pd.DataFrame(historical_seed42)

finbert_08_summary = (
    BASE_CHECKPOINT_ROOT
    / "finbert_target_finetuned_seed42"
    / "results"
    / "finbert_target_finetune_summary.csv"
)

if finbert_08_summary.exists():
    f08 = pd.read_csv(finbert_08_summary)

    if len(f08) != 1:
        raise ValueError(
            f"08 FinBERT summary tek satır bekleniyordu: {finbert_08_summary}"
        )

    seed42_df = pd.concat(
        [
            seed42_df,
            pd.DataFrame([
                {
                    "model_key": "finbert",
                    "model": "Fine-tuned FinBERT",
                    "seed": 42,
                    "accuracy": float(f08.iloc[0]["test_accuracy"]),
                    "f1_macro": float(f08.iloc[0]["test_f1_macro"]),
                    "f1_weighted": float(f08.iloc[0]["test_f1_weighted"]),
                }
            ])
        ],
        ignore_index=True,
    )

    print("08 FinBERT seed=42 sonucu bulundu ve eklendi.")
else:
    print(
        "UYARI: Fine-tuned FinBERT seed=42 summary bulunamadı.\n"
        "Önce 08_finetune_finbert_target_dataset.ipynb çalıştırılırsa "
        "FinBERT için 3-seed mean/std de hesaplanabilir.\n"
        "Şimdilik diğer modeller 3 seed, FinBERT yalnızca yeni seedlerle özetlenir."
    )

display(seed42_df.round(6))


In [ ]:
# ============================================================
# 10) 3-SEED ORTALAMA ± STANDART SAPMA
# ============================================================

new_compact = new_seed_results_df[
    ["model_key", "model", "seed", "accuracy", "f1_macro", "f1_weighted"]
].copy()

all_seed_results_df = pd.concat(
    [seed42_df, new_compact],
    ignore_index=True,
)

all_seed_results_df = (
    all_seed_results_df
    .sort_values(["model", "seed"])
    .reset_index(drop=True)
)

all_seed_results_path = MULTISEED_ROOT / "multiseed_results_with_seed42.csv"
all_seed_results_df.to_csv(
    all_seed_results_path,
    index=False,
    encoding="utf-8-sig",
)

summary_df = (
    all_seed_results_df
    .groupby(["model_key", "model"], as_index=False)
    .agg(
        n_seeds=("seed", "count"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
        f1_weighted_mean=("f1_weighted", "mean"),
        f1_weighted_std=("f1_weighted", "std"),
    )
)

summary_df = summary_df.sort_values(
    "f1_macro_mean",
    ascending=False,
).reset_index(drop=True)

summary_path = MULTISEED_ROOT / "multiseed_mean_std_summary.csv"
summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

def mean_std_text(mean, std):
    if pd.isna(std):
        return f"{mean:.4f}"
    return f"{mean:.4f} ± {std:.4f}"

pretty_df = summary_df.copy()
pretty_df["Accuracy mean ± std"] = [
    mean_std_text(m, s)
    for m, s in zip(
        pretty_df["accuracy_mean"],
        pretty_df["accuracy_std"],
    )
]
pretty_df["Macro-F1 mean ± std"] = [
    mean_std_text(m, s)
    for m, s in zip(
        pretty_df["f1_macro_mean"],
        pretty_df["f1_macro_std"],
    )
]
pretty_df["Weighted-F1 mean ± std"] = [
    mean_std_text(m, s)
    for m, s in zip(
        pretty_df["f1_weighted_mean"],
        pretty_df["f1_weighted_std"],
    )
]

print("\n" + "#" * 110)
print("MULTI-SEED SUMMARY")
print("#" * 110)

display(
    pretty_df[
        [
            "model",
            "n_seeds",
            "Accuracy mean ± std",
            "Macro-F1 mean ± std",
            "Weighted-F1 mean ± std",
        ]
    ]
)

print("\nKaydedildi:")
print("Yeni seed sonuçları :", AGGREGATE_RESULTS_PATH)
print("Tüm seed sonuçları :", all_seed_results_path)
print("Mean/std summary    :", summary_path)

print("\nNot:")
print("- n_seeds=3 olan satırlar tezde doğrudan mean ± std olarak kullanılabilir.")
print("- Fine-tuned FinBERT n_seeds=2 görünüyorsa önce notebook 08'i çalıştır.")


## Çalıştırma sırası

Bu notebooktan önce:

1. `04_train_plain_sentiment_models.ipynb` tamamlanmış olmalı.
2. `08_finetune_finbert_target_dataset.ipynb` çalıştırılmış olmalı.

Sonra bu notebooku **Run All** ile başlat.

CPU kullanıyorsan 8 yeni training run uzun sürebilir. Notebook güvenli şekilde tekrar çalıştırılabilir;
bitmiş run'ları atlar, yarım kalan run için checkpoint varsa devam eder.

## Bana göndermen gerekenler

Çalışma tamamlanınca şu iki dosyayı yükle:

- `checkpoints/financial_sentiment_multi_model/multiseed/multiseed_mean_std_summary.csv`
- `checkpoints/financial_sentiment_multi_model/multiseed/multiseed_results_with_seed42.csv`

Mümkünse ayrıca `multiseed` klasöründeki `test_predictions.csv` dosyalarını da sakla;
istatistiksel anlamlılık notebookunda kullanılacaklar.
